# FAME Database Inspection & Auditing Tools
This notebook provides a suite of tools to quickly audit, sample, and query the compiled FAME DuckDB database.

It uses `Ibis` to push computations directly to DuckDB, meaning queries execute in milliseconds and use almost zero Python memory.

In [ ]:
from pathlib import Path
from dataclasses import dataclass

# Define the structure clearly
@dataclass
class Dirs:
    data_dir: Path = None       # type: ignore
    output_dir: Path = None     # type: ignore
    input_dir: Path = None      # type: ignore
dirs = Dirs()

try:
    from f_0_dirs import get_data_dirs
    dirs = get_data_dirs()
    raise ImportError("Testing ImportError for demonstration purposes") 

# For easy access purposes, if this script is run directly in a folder with the databases
# and without the dir import modules, then just set it to current_dir
except ImportError:
    try:
        current_dir = Path(__file__).parent
    except NameError:
        current_dir = Path.cwd()
    dirs = Dirs(output_dir=current_dir, data_dir=current_dir, input_dir=current_dir)

print(dirs)

Dirs(data_dir=WindowsPath('c:/Users/lazyst/Files/ucl/Dissertation/build/src'), output_dir=WindowsPath('c:/Users/lazyst/Files/ucl/Dissertation/build/src'))


In [52]:
import pandas as pd
import ibis
import ibis.selectors as s

# 1. Setup paths
db_path = dirs.output_dir / "fame_data.duckdb"

# 2. Connect to the database
con = ibis.duckdb.connect(str(db_path))

# 3. Configure Pandas display for easier reading
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 50)

print(f"✅ Successfully connected to: {db_path}")

✅ Successfully connected to: c:\Users\lazyst\Files\ucl\Dissertation\build\src\fame_data.duckdb


## 1. High-Level Database Overview
Quickly check which tables exist in the database and their total row counts.

In [2]:
tables = con.list_tables()
print("📊 Database Tables Overview:\n" + "-"*30)

for table_name in tables:
    t = con.table(table_name)
    # Fast row counting pushed down to DuckDB
    row_count = t.count().execute()
    col_count = len(t.columns)
    print(f"[{table_name}]")
    print(f"   Rows: {row_count:,} | Columns: {col_count}")
print("-" * 30)

📊 Database Tables Overview:
------------------------------
[fame_derived]
   Rows: 2,530,879 | Columns: 7
[fame_fixed]
   Rows: 2,530,879 | Columns: 29
[fame_yearly]
   Rows: 31,568,718 | Columns: 25
[lars_fixed]
   Rows: 452,638 | Columns: 26
[lars_yearly]
   Rows: 2,378,089 | Columns: 35
------------------------------


In [ ]:
import ibis

out_file = dirs.output_dir / "duckdb_tables.md"
con = ibis.duckdb.connect(str(dirs.output_dir / "fame_data.duckdb"))
interesting_tables = ["fame_fixed", "fame_derived", "fame_yearly", "lars_fixed", "lars_yearly"]
with open(out_file, "w") as f:
    f.write("# Tables in DuckDB database\n\n")
    for table in interesting_tables:
        f.write(f"## {table}\n\n")
        f.write(f"### Number of rows: {con.table(table).count().execute():,}\n\n")
        f.write(f"### Schema:\n\n```\n{con.table(table).schema()}\n```\n\n")
        f.write(f"### Head of table:\n\n```\n{con.table(table).head().execute()}\n```\n\n")
print(f"✅ Successfully listed tables and their heads in: {out_file}")

✅ Successfully listed tables and their heads in: C:\Users\lazyst\Files\ucl\Dissertation\build\output\duckdb_tables.md


## 2. Fast Random Sampling (Reservoir Sampling)
Extract a random subset of rows for visual inspection. We use DuckDB's native reservoir sampling (`USING SAMPLE X ROWS`) so it returns instantly, even on tables with millions of rows, without doing a full table scan.

In [12]:
target_table = "fame_yearly"
sample_frac = 0.001  # Sample fraction for random sampling

# Native Ibis sampling using positional row count and seed parameter
table = con.table(target_table)
row_count = table.count().execute()
df_sample = table.sample(sample_frac, seed=12345).execute()

print(f"🎲 Random sample of {row_count:,} rows from '{target_table}':")
display(df_sample)

🎲 Random sample of 31,568,718 rows from 'fame_yearly':


,registered_number,year,consolidated,turnover,shareholders_funds,profit_loss_pretax,employees,tangibles,tangibles_land_and_buildings,tangibles_land_freehold,tangibles_land_leasehold,fixed_other,intangibles,fixed_total,liabilities,total_assets,liabilites_lt,cos,dividends,r_and_d,remuneration_employees,wages,social_security_costs,pensions_costs,ebitda
0,04539608,2023,False,NaN,50.046,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,04742309,2011,False,NaN,32.833,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,04980124,2022,False,NaN,890.829,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,05067241,2006,False,NaN,2.968,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,05500567,2009,False,NaN,82.033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31523,06093533,2013,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-40.236000,541.693000,-449.632,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31524,08147310,2019,None,NaN,NaN,NaN,NaN,6.188000,NaN,NaN,NaN,6.188,NaN,6.188000,-13.366000,15.156000,-1.521,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31525,IE483682,2019,None,NaN,NaN,NaN,NaN,3.353032,NaN,NaN,NaN,NaN,NaN,3.353032,-13.212758,74.307717,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31526,03610218,2023,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-138.176


## 3. Specific Lookup

### Firm-specific (Search by Name or ID)
Find all fixed and derived attributes for a specific company using partial string matching (case-insensitive).

In [16]:
search_term = "TESCO"  # Can be a partial name or a Registered Number
table_fixed = con.table("fame_fixed")

# Filter where company_name contains the search term OR exactly matches registered_number
search_query = table_fixed.filter(
    table_fixed["company_name"].upper().contains(search_term.upper()) |
    (table_fixed["registered_number"] == search_term)
)

# Pull the top 5 matches
df_search_results = search_query.head(10).execute()

print(f"🔍 Top search results for '{search_term}':")
display(df_search_results[["registered_number", "company_name", "ro_address", "primary_trading_address"]])

🔍 Top search results for 'TESCO':


,registered_number,company_name,ro_address,primary_trading_address
0,10645827,BITTESCOMBE MANOR ESTATE LTD,"Bittescombe Manor Upton, Taunton, Somerset, TA...",NaN
1,10645827,BITTESCOMBE MANOR ESTATE LTD,"Bittescombe Manor Upton, Taunton, Somerset, TA...",NaN
2,11797367,TESCON CONSULTING LTD,"c/o Johnston Carmichael Office G, Ground Floor...",NaN
3,12302279,INTESCOM LTD,"34 New House, 67-68 Hatton Garden, London, EC1...",NaN
4,03527086,SITESCOPE LIMITED,"Unit 5-7, Abbey Court, Eagle Way, Sowton Indus...",NaN
5,05769298,TESCO TECH SUPPORT LIMITED,"Tesco House, Delamare Road, Cheshunt, Waltham ...","Tesco House, Delamare Road, Cheshunt, Waltham ..."
6,07374250,WHITESCOMPUTERSERVICES LIMITED,"89 The Mount, Ringwood, Hampshire, BH24 1XZ","89 The Mount, Ringwood, Hampshire, BH24 1XZ"
7,06353207,TESCO.EU.COM LIMITED,"5 Whitmore Crescent, Chelmsford, Essex, CM2 6YN","5 Whitmore Crescent, Chelmsford, Essex, CM2 6YN"
8,03091560,TESCOM (UK) SOFTWARE SYSTEMS TESTING LIMITED,"Flat 3, 80 The Lakes, Larkfield, Aylesford, Ke...",NaN
9,04460370,ELITESCORE LIMITED,"19 The Maltings, Tingewick, Buckingham, Buckin...",NaN


### Industry-specific (by 2-digit SIC or 6-digit
- 6-digit using primary_uk_sic_2007_code in fame_fixed

In [ ]:
# ### Industry-specific (by 2-digit SIC or 6-digit
# - 6-digit using primary_uk_sic_2007_code in fame_fixed
import json
import ibis
import pandas as pd

# Read build/input/SIC_priorities.json
search_inds = []

get_from_json = False
if get_from_json:
    with open(dirs.input_dir / "SIC_priorities.json", "r") as f:
        sic_priorities = json.load(f)
        search_inds = [entry.get("Division") for entry in sic_priorities if entry.get("Errored") == True]
else:
    search_inds = ["70", "74"]

# Ensure strings are properly formatted (e.g. padded with zeros) just in case
search_inds_str = [str(x).zfill(2) for x in search_inds if x is not None]
print(f"Searching for following SIC divisions individually: {search_inds_str}")

# Connect to the tables in the database
t_fixed = con.table("fame_fixed")
t_derived = con.table("fame_derived")
t_yearly = con.table("fame_yearly")

# List to accumulate our row records
results = []

for ind in search_inds_str:
    # 1. Filter the fame_derived table for the specific SIC division
    regex_pattern = rf'\b{ind}\b'
    filtered_derived = t_derived.filter(
        t_derived.industry_codes.re_search(regex_pattern)
    )

    # 2. Execute the count for fame_derived
    derived_count = filtered_derived.count().execute()

    # 3. Filter fame_fixed using a semi-join
    fixed_count = t_fixed.semi_join(filtered_derived, "registered_number").count().execute()

    # 4. Filter fame_yearly using a semi-join
    yearly_count = t_yearly.semi_join(filtered_derived, "registered_number").count().execute()

    # Append the counts for this specific division to our results list
    results.append({
        "SIC_Division": ind,
        "fame_derived": derived_count,
        "fame_fixed": fixed_count,
        "fame_yearly": yearly_count
    })

# Output the results as a formatted table
output_table = pd.DataFrame(results)

print("📊 Matching Rows by Table for Each SIC Division:")
display(output_table) # use print(output_table) if you aren't in a Jupyter environment

Searching for following SIC divisions individually: ['70', '74']
📊 Matching Rows by Table for Each SIC Division:


,SIC_Division,fame_derived,fame_fixed,fame_yearly
0,70,0,0,0
1,74,235756,274769,3503927


## 4. Time-Series Construction (Joining Fixed & Yearly)
Combine the static company metadata with its longitudinal financial performance. This demonstrates the relational integrity of the `registered_number` primary key.

In [20]:
# Pick a specific company ID to track over time
selected_ids = df_sample["registered_number"].head().tolist()

t_fixed = con.table("fame_fixed")
t_yearly = con.table("fame_yearly")

# 1. Filter both tables to the target ID
firm_fixed = t_fixed.filter(t_fixed["registered_number"].isin(selected_ids))
firm_yearly = t_yearly.filter(t_yearly["registered_number"].isin(selected_ids))

# 2. Left join yearly financials onto the fixed metadata
firm_history = firm_yearly.left_join(firm_fixed, "registered_number")

# 3. Select a curated subset of columns to display
comprehensive_view = firm_history.select(
    "registered_number",
    "company_name",
    "year",
    # Safely select financial columns if they exist in the DB
    s.contains("turnover"),
    s.contains("profit_loss_pretax"),
    s.contains("employees")
).order_by("year") # Order chronologically

display(comprehensive_view.execute())

,registered_number,company_name,year,turnover,profit_loss_pretax,employees,remuneration_employees
0,05067241,SOUTH DEVON CHILLI FARM LTD,2006,NaN,NaN,NaN,NaN
1,04980124,EAST AVERCOMBE FARM LIMITED,2006,NaN,NaN,NaN,NaN
2,04980124,EAST AVERCOMBE FARM LIMITED,2006,NaN,NaN,NaN,NaN
3,05067241,SOUTH DEVON CHILLI FARM LTD,2006,NaN,NaN,NaN,NaN
4,04742309,R J BROWN LIMITED,2006,69.697,20.345,NaN,NaN
...,...,...,...,...,...,...,...
371,05067241,SOUTH DEVON CHILLI FARM LTD,2024,NaN,NaN,16.0,NaN
372,05500567,NEWHOUSE MILL LIMITED,2024,NaN,NaN,20.0,NaN
373,05500567,NEWHOUSE MILL LIMITED,2024,NaN,NaN,20.0,NaN
374,05067241,SOUTH DEVON CHILLI FARM LTD,2024,NaN,NaN,16.0,NaN


## 5. Summary Statistics & Aggregations
Generate high-level analytical cuts (e.g., counting the number of records per year, or assessing data coverage).

In [ ]:
t_yearly = con.table("fame_yearly")

# Aggregate the number of financial records available per year
# Print numbers of available records for every financial column
yearly_distribution = (
    t_yearly
    .group_by("year")
    .aggregate(
        turnover=t_yearly["turnover"].count(),
        pnl=t_yearly["profit_loss_pretax"].count(),
        employees=t_yearly["employees"].count(),
        tangibles=t_yearly["tangibles"].count(),
        t_lab=t_yearly["tangibles_land_and_buildings"].count(),
        t_land_free=t_yearly["tangibles_land_freehold"].count(),
        t_land_lease=t_yearly["tangibles_land_leasehold"].count(),
        fixed_other=t_yearly["fixed_other"].count(),
        intangibles=t_yearly["intangibles"].count(),
        fixed_total=t_yearly["fixed_total"].count(),
        liabilities=t_yearly["liabilities"].count(),
        t_assets=t_yearly["total_assets"].count(),
        liabilites_lt=t_yearly["liabilites_lt"].count(),
        cos=t_yearly["cos"].count(),
        dividends=t_yearly["dividends"].count(),
        r_and_d=t_yearly["r_and_d"].count(),
        renum=t_yearly["remuneration_employees"].count(),
        wages=t_yearly["wages"].count(),
        ss_cost=t_yearly["social_security_costs"].count(),
        pension_cost=t_yearly["pensions_costs"].count(),
        ebitda=t_yearly["ebitda"].count(),
        all=t_yearly.count()
    )
    .order_by(ibis.desc("year"))
)

print("📈 Data coverage by year:")
# Format table with commas for readability and display
count_table: pd.DataFrame = yearly_distribution.execute()
count_table_formatted = count_table.copy().reset_index(drop=True)
for col in count_table_formatted.columns:
    if col != "year":
        count_table_formatted[col] = count_table_formatted[col].apply(lambda x: f"{x:,}")
display(count_table_formatted)

📈 Data coverage by year:


,year,turnover,pnl,employees,tangibles,t_lab,t_land_free,t_land_lease,fixed_other,intangibles,fixed_total,liabilities,t_assets,liabilites_lt,cos,dividends,r_and_d,renum,wages,ss_cost,pension_cost,ebitda,all
0,2025,563,651,"6,467","3,834",252,159,106,"2,712",340,"5,126","5,900","8,183","2,930",2,0,0,0,0,0,0,2,"19,735"
1,2024,"37,479","42,864","443,744","310,820","20,887","15,424","6,758","228,838","21,706","382,039","475,405","589,019","219,312","12,458","4,300",689,"22,567","16,715","13,769","14,087","41,737","1,317,616"
2,2023,"88,390","105,935","769,777","589,209","48,499","33,788","17,697","426,616","53,760","741,649","912,461","1,134,025","433,247","36,076","11,857","2,499","53,871","44,188","36,988","36,846","103,200","2,463,669"
3,2022,"87,839","105,490","750,995","576,406","48,471","33,260","18,274","417,815","53,172","722,580","889,182","1,103,744","426,217","35,535","11,443","2,415","53,151","43,490","36,505","36,118","102,601","2,399,731"
4,2021,"86,416","103,397","743,328","562,847","47,143","32,175","17,801","407,814","51,999","698,155","874,973","1,079,886","416,226","33,495","11,012","2,269","52,785","42,800","35,561","34,957","100,708","2,346,092"
5,2020,"85,971","101,479","720,047","546,883","46,889","32,333","17,369","398,892","50,487","667,417","860,434","1,042,671","348,203","32,357","10,923","2,162","51,672","44,470","34,810","33,755","98,789","2,266,969"
6,2019,"84,165","98,758","537,253","527,007","48,317","35,076","16,660","395,966","49,262","638,399","821,531","992,964","288,078","31,207","11,920","2,125","50,033","49,440","33,587","31,707","95,885","2,161,520"
7,2018,"82,713","96,332","442,900","508,591","45,380","33,316","15,087","367,969","49,073","610,665","783,797","943,884","271,784","30,752","12,620","1,847","50,594","50,246","33,132","30,524","94,875","2,058,942"
8,2017,"86,457","100,467","365,461","495,324","44,228","32,805","14,235","353,054","50,117","582,683","762,875","912,288","256,805","31,806","14,425","1,785","54,099","53,739","34,015","29,580","100,188","1,999,149"
9,2016,"76,218","87,675","191,111","475,991","25,280","16,508","11,108","414,345","61,633","562,295","728,574","870,865","217,636","30,715","16,698","1,367","54,108","53,544","37,054","29,131","86,840","1,899,136"


## 6. Exporting Queries to Excel
Any queried subset of data can be instantly dumped into an Excel file for offline review.

In [29]:
import pandas as pd

approach = 'random' # first

# Dump a random sample of 500 rows from each table to a single .xlsx file in /tmp
row_count = 2000
out_file_raw = dirs.output_dir / "df_raw_sample.xlsx"
desired_tables = ["fame_derived", "fame_fixed", "fame_yearly", "lars_fixed", "lars_yearly"]

# but we want to have 5 separate tabs in the same excel file, one for each table, with the table name as the tab name
con = ibis.duckdb.connect(str(dirs.output_dir / "fame_data.duckdb"))
with pd.ExcelWriter(out_file_raw, engine='openpyxl') as writer:
    # df_raw_head.to_excel(writer, sheet_name='df_raw', index=False)
    for table in desired_tables:
        if approach == 'first':
            con.table(table).head(row_count).execute().to_excel(writer, sheet_name=table, index=False)
        elif approach == 'random':
            nb_rows = con.table(table).count().execute()
            fraction = row_count / nb_rows if nb_rows > row_count else 1.0
            df_table_rd = con.table(table).sample(fraction, seed=12345).execute()
            df_table_rd.to_excel(writer, sheet_name=table, index=False)
        
print(f"✅ Successfully dumped a random {row_count} rows of df_raw to: {out_file_raw}")

✅ Successfully dumped a random 2000 rows of df_raw to: C:\Users\lazyst\Files\ucl\Dissertation\build\output\df_raw_sample.xlsx
